In [ ]:

````xml
<!-- filepath: m:\unsloth_clean_for_docs_tests\unsloth_test_3_14_26_pi_day\unsloth\studio\GPT_OSS_20B_Training_Debug_Floaty.ipynb -->
<VSCode.Cell language="markdown">
# GPT-OSS-20B QLoRA Training Debug Notebook
## Issue: Loss Collapse to ~0 within 50-300 steps

**Discord User:** floaty  
**Hardware:** RTX 3090 24GB VRAM, Ubuntu 24.04  
**Dataset:** HuggingFaceH4/Multilingual-Thinking  
**Model:** unsloth/gpt-oss-20b with 4-bit quantization  

### Key Observations:
1. **Initial loss starts at ~9.874** (vs expected ~2.79)
2. **Loss drops to ~1e-5** within 200-300 steps
3. **Inference produces token loops** after training
4. **Flash Attention 2 appears broken** on user's system(s)
5. **Issue persists across multiple machines** with same FA2 warning
6. **Official dataset from HF still causes issue** (not a dataset corruption problem)

### Hypothesis:
The abnormally high initial loss (~9.874 vs ~2.79) suggests the model may not be properly initialized or there's an attention mechanism issue causing gradient instability.

---
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 1: Installation with diagnostics
import os, importlib.util, sys

print("=" * 60)
print("DIAGNOSTIC: Environment Setup")
print("=" * 60)

# Check if we need to install
needs_install = importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()) or importlib.util.find_spec("unsloth") is None

if needs_install:
    print("📦 Installing dependencies...")
    !pip install --upgrade -qqq uv

    if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
        try:
            import numpy, PIL
            _numpy = f"numpy=={numpy.__version__}"
            _pil = f"pillow=={PIL.__version__}"
        except:
            _numpy = "numpy"
            _pil = "pillow"

        !uv pip install -qqq \
            "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
            "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
            "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
            git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
    elif importlib.util.find_spec("unsloth") is None:
        !uv pip install -qqq unsloth

    !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

    print("\n⚠️ Installation complete! Please restart the runtime: Runtime -> Restart session")
    print("Then run from the next cell.")
else:
    print("✅ Dependencies already installed")

# Print version info
print("\n" + "=" * 60)
print("DIAGNOSTIC: Package Versions")
print("=" * 60)
try:
    import torch
    import transformers
    import unsloth
    import trl
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA Version: {torch.version.cuda}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Transformers: {transformers.__version__}")
    print(f"Unsloth: {unsloth.__version__ if hasattr(unsloth, '__version__') else 'Unknown'}")
    print(f"TRL: {trl.__version__}")
except Exception as e:
    print(f"❌ Error checking versions: {e}")
print("=" * 60)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 2: Load model with diagnostics
from unsloth import FastLanguageModel
import torch

print("\n" + "=" * 60)
print("DIAGNOSTIC: Model Loading")
print("=" * 60)

max_seq_length = 3072
dtype = None  # Auto-detect

print(f"Max sequence length: {max_seq_length}")
print(f"Data type: {dtype} (auto-detect)")
print(f"Loading model: unsloth/gpt-oss-20b")
print("=" * 60 + "\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype,
    max_seq_length = max_seq_length,
    load_in_4bit = True,  # 4-bit quantization
    full_finetuning = False,
    # token = "hf_...",  # Uncomment if using gated models
)

print("\n" + "=" * 60)
print("DIAGNOSTIC: Model Loaded Successfully")
print("=" * 60)
print(f"Model config:")
print(f"  - Hidden size: {model.config.hidden_size}")
print(f"  - Num layers: {model.config.num_hidden_layers}")
print(f"  - Num attention heads: {model.config.num_attention_heads}")
print(f"  - Vocab size: {model.config.vocab_size}")

# Check Flash Attention status
try:
    if hasattr(model.config, '_attn_implementation'):
        print(f"  - Attention implementation: {model.config._attn_implementation}")
except:
    pass

print("=" * 60)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 3: Apply LoRA with diagnostics
print("\n" + "=" * 60)
print("DIAGNOSTIC: Applying LoRA Adapter")
print("=" * 60)

lora_config = {
    "r": 8,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    "lora_alpha": 16,
    "lora_dropout": 0,
    "bias": "none",
    "use_gradient_checkpointing": "unsloth",
    "random_state": 3407,
    "use_rslora": False,
    "loftq_config": None,
}

print("LoRA configuration:")
for key, value in lora_config.items():
    print(f"  - {key}: {value}")
print("=" * 60 + "\n")

model = FastLanguageModel.get_peft_model(model, **lora_config)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = 100 * trainable_params / total_params

print("\n" + "=" * 60)
print("DIAGNOSTIC: LoRA Applied Successfully")
print("=" * 60)
print(f"Trainable parameters: {trainable_params:,} ({trainable_pct:.4f}%)")
print(f"Total parameters: {total_params:,}")
print("=" * 60)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 4: Load and prepare dataset with diagnostics
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt, train_on_responses_only

print("\n" + "=" * 60)
print("DIAGNOSTIC: Loading Dataset")
print("=" * 60)

dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")

print(f"✅ Dataset loaded: {len(dataset)} examples")
print(f"Dataset features: {dataset.features.keys()}")
print(f"First example keys: {dataset[0].keys() if len(dataset) > 0 else 'N/A'}")
print("=" * 60)

# Standardize dataset format
print("\n" + "=" * 60)
print("DIAGNOSTIC: Standardizing Dataset Format")
print("=" * 60)

dataset = standardize_sharegpt(dataset)
print("✅ Dataset standardized to ShareGPT format")
print("=" * 60)

# Format prompts
print("\n" + "=" * 60)
print("DIAGNOSTIC: Applying Chat Template")
print("=" * 60)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    # NOTE: NOT using reasoning_effort parameter (per official notebook)
    texts = [tokenizer.apply_chat_template(
        convo, 
        tokenize=False, 
        add_generation_prompt=False
    ) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✅ Chat template applied to {len(dataset)} examples")
print(f"Number of training items: {len(dataset)}")
print("=" * 60)

# Show sample
print("\n" + "=" * 60)
print("DIAGNOSTIC: Sample Formatted Example")
print("=" * 60)
print(dataset[0]["text"][:500] + "..." if len(dataset[0]["text"]) > 500 else dataset[0]["text"])
print("=" * 60)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 5: Create train/test split with diagnostics
print("\n" + "=" * 60)
print("DIAGNOSTIC: Creating Train/Test Split")
print("=" * 60)

new_dataset = dataset.train_test_split(
    test_size=0.01,  # 1% for test
    shuffle=True,
    seed=3407,
)

train_dataset = new_dataset["train"]
eval_dataset = new_dataset["test"]

print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")
print("=" * 60)

# Show first training example
print("\n" + "=" * 60)
print("DIAGNOSTIC: First Training Example")
print("=" * 60)
print(train_dataset[0]["text"][:1000] + "..." if len(train_dataset[0]["text"]) > 1000 else train_dataset[0]["text"])
print("=" * 60)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 6: Setup trainer with diagnostics
from trl import SFTConfig, SFTTrainer

print("\n" + "=" * 60)
print("DIAGNOSTIC: Configuring Trainer")
print("=" * 60)

training_args = SFTConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    num_train_epochs=1,  # Full training run
    # max_steps=100,  # Uncomment for quick test
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs_floaty_debug",
    report_to="none",
    bf16=True,  # ⚠️ CRITICAL: Enable bf16 for RTX 3090
    
    save_strategy="steps",
    save_steps=50,
    
    # Evaluation settings
    fp16_full_eval=True,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=50,
)

print("Training configuration:")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  - Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  - Learning rate: {training_args.learning_rate}")
print(f"  - Warmup steps: {training_args.warmup_steps}")
print(f"  - Epochs: {training_args.num_train_epochs}")
print(f"  - BF16: {training_args.bf16}")
print(f"  - Optimizer: {training_args.optim}")
print("=" * 60)

# Create trainer
print("\n" + "=" * 60)
print("DIAGNOSTIC: Creating Trainer")
print("=" * 60)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

# NOTE: NOT using train_on_responses_only() per Discord debugging
# GPT-OSS uses two-channel format (analysis + final) and masking breaks training

print("✅ Trainer created successfully")
print("=" * 60)

# Print sample of what the model will see
print("\n" + "=" * 60)
print("DIAGNOSTIC: Sample Tokenized Input")
print("=" * 60)
sample_idx = 5
print(f"Example {sample_idx} input_ids (first 200 tokens):")
print(tokenizer.decode(trainer.train_dataset[sample_idx]["input_ids"][:200]))
print("=" * 60)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 7: Custom training loop with enhanced diagnostics
print("\n" + "=" * 60)
print("DIAGNOSTIC: Starting Training with Enhanced Logging")
print("=" * 60)
print("⚠️ Watch for:")
print("  1. Initial loss (~2.79 is normal, ~9.87 indicates problem)")
print("  2. Loss progression (should decrease gradually)")
print("  3. Gradient norms (should be stable)")
print("  4. Evaluation loss (should track training loss)")
print("=" * 60 + "\n")

# Hook to capture initial loss
class InitialLossDiagnostic:
    def __init__(self):
        self.first_loss = None
        self.first_10_losses = []
        
initial_diagnostic = InitialLossDiagnostic()

def log_metrics_callback(metrics):
    """Enhanced logging callback"""
    if 'loss' in metrics and len(initial_diagnostic.first_10_losses) < 10:
        initial_diagnostic.first_10_losses.append(metrics['loss'])
        if initial_diagnostic.first_loss is None:
            initial_diagnostic.first_loss = metrics['loss']
            print(f"\n🔍 FIRST LOSS: {metrics['loss']:.6f}")
            if metrics['loss'] > 5.0:
                print("⚠️ WARNING: Initial loss is abnormally high! Expected ~2.79")
                print("   This may indicate an attention mechanism or initialization issue.")
            else:
                print("✅ Initial loss looks normal")
            print()

# Add callback
from transformers import TrainerCallback

class DiagnosticCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            log_metrics_callback(logs)

trainer.add_callback(DiagnosticCallback())

# Start training
print("🚀 Starting training...")
print("=" * 60 + "\n")

trainer.train()

print("\n" + "=" * 60)
print("DIAGNOSTIC: Training Complete")
print("=" * 60)
print(f"First loss: {initial_diagnostic.first_loss}")
print(f"First 10 losses: {initial_diagnostic.first_10_losses}")
print("=" * 60)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 8: Test inference with diagnostics
print("\n" + "=" * 60)
print("DIAGNOSTIC: Testing Inference")
print("=" * 60)

FastLanguageModel.for_inference(model)

# Test prompt
test_prompt = "Why is the sky blue?"

messages = [
    {"role": "user", "content": test_prompt}
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

print(f"Test prompt: {test_prompt}")
print("Generating response...\n")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)

print("=" * 60)
print("GENERATED OUTPUT:")
print("=" * 60)
print(generated_text)
print("=" * 60)

# Check for token looping
words = generated_text.split()
if len(words) > 10:
    last_10 = words[-10:]
    unique_last_10 = set(last_10)
    if len(unique_last_10) < 3:
        print("\n⚠️ WARNING: Detected potential token looping!")
        print(f"   Last 10 words: {last_10}")
    else:
        print("\n✅ No obvious token looping detected")
print("=" * 60)
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 9: Save model
print("\n" + "=" * 60)
print("DIAGNOSTIC: Saving Model")
print("=" * 60)

output_dir = "./gpt-oss-20b-floaty-debug"

print(f"Saving LoRA adapter to: {output_dir}")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\nSaving merged model (mxfp4) to: {output_dir}-merged")
model.save_pretrained_merged(f"{output_dir}-merged", tokenizer, save_method="mxfp4")

print("\n✅ Model saved successfully")
print("=" * 60)
</VSCode.Cell>
<VSCode.Cell language="markdown">
---

## Debugging Checklist

Compare your local output to this notebook:

### ✅ Normal Behavior:
- [ ] Initial loss starts around **2.5-3.0**
- [ ] Loss decreases gradually over 200+ steps
- [ ] Evaluation loss tracks training loss
- [ ] Gradient norms stay stable (1-10 range)
- [ ] Flash Attention message shows `Xformers = None` or proper FA2
- [ ] Generated text is coherent (no looping)

### ⚠️ Problem Indicators:
- [ ] Initial loss starts **above 5.0** (especially 9.87+)
- [ ] Loss drops to near-zero within 50-300 steps
- [ ] Evaluation loss stays very high while training loss drops
- [ ] Gradient norms spike or become unstable
- [ ] Flash Attention shows `broken` or falls back to Xformers
- [ ] Generated text loops tokens

---

## Next Steps if Issue Persists:

1. **Check CUDA/Torch compatibility:**
   ```python
   import torch
   print(f"PyTorch: {torch.__version__}")
   print(f"CUDA: {torch.version.cuda}")
   print(f"Compiled with CUDA: {torch.version.cuda}")
   ```

2. **Try forcing attention implementation:**
   ```python
   model, tokenizer = FastLanguageModel.from_pretrained(
       model_name="unsloth/gpt-oss-20b",
       max_seq_length=3072,
       load_in_4bit=True,
       attn_implementation="sdpa",  # or "flash_attention_2"
   )
   ```

3. **Test with smaller batch/lr:**
   ```python
   training_args = SFTConfig(
       per_device_train_batch_size=1,
       gradient_accumulation_steps=2,  # Reduce from 4
       learning_rate=1e-4,  # Reduce from 2e-4
       # ...rest of config
   )
   ```

4. **Check for gradient issues:**
   Add to training loop:
   ```python
   for name, param in model.named_parameters():
       if param.requires_grad and param.grad is not None:
           print(f"{name}: grad_norm={param.grad.norm()}")
   ```

---

**For questions or issues, contact Leo [UnAI] on Discord**
</VSCode.Cell>
````